# Adding refractoriness to the Izhikevich neuron model in NESTML

Introduction
------------

The aim of this exercise is to obtain familiarity with NESTML by adding a refractory mechanism to the Izhikevich spiking neuron model [1]. An absolute refractory period is a short interval of time right after a spike has occurred, in which a neuron is precluded from firing another spike. This places an upper limit on the firing rate of the neuron: if the refractory period is $T_\mathrm{refr}$, then the maximum rate is $1/T_\mathrm{refr}$.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import nest
import numpy as np

from pynestml.codegeneration.nest_code_generator_utils import NESTCodeGeneratorUtils
from pynestml.codegeneration.nest_tools import NESTTools

The Izhikevich model
--------------------

A simple model for spiking neurons that nevertheless can exhibit a wide variety of dynamical behaviour, depending on its parameter values [1]. It is defined as follows:

\begin{align}
\frac{dv}{dt} &= 0.04 v^2 + 5 v + 140 - u + I + s\\[.5cm]
\frac{du}{dt} &= a (b v - u)
\end{align}

where

\begin{align}
s(t) = \sum_i w_i \delta(t - t_i)
\end{align}

is the weighted input spike train, here modelled as a series of Dirac delta pulses.

The output spikes of the neuron are described by the condition:

\begin{align}
   &\text{if}\;\; v \geq V_\mathrm{th}:\\
   &\;\;\;\; v \text{ is set to } c\\
   &\;\;\;\; u \text{ is incremented by } d\\
\end{align}

Example parameters for regular spiking (the meaning of these parameters is described in detail in the paper [1]; see also Task 2 below): 

\begin{align}
a&=0.02\\
b&=0.2\\
c&=-65~\text{mV}\\
d&=8
\end{align}

### The Izhikevich model in NESTML

If you are running this notebook locally or cloned the repository, the NESTML model ``izhikevich_neuron.nestml`` is contained in the subdirectory ``models/neurons``. You can open NESTML model files in your favourite code editor (check https://github.com/nest/nestml/ for syntax highlighting support). In case you are running this notebook in JupyerLab, you can edit the model files directly in your browser via the “File browser” panel on the left.

The model can be found on our git repository at: https://github.com/nest/nestml/blob/main/models/neurons/izhikevich_neuron.nestml

### NESTML code generation

When NESTML is invoked, the C++ code is generated for the models, and then built (compiled) as a NEST extension module, which is then loaded into the NEST kernel at runtime using ``nest.Install()``.

In [ ]:
# generate and build code
module_name, neuron_model_name = \
    NESTCodeGeneratorUtils.generate_code_for("../../../models/neurons/izhikevich_neuron.nestml",
                                             logging_level="ERROR")  # try "INFO" or "DEBUG" for more debug information)

Check the generated log output for any potential error messages or warnings.

### Instantiate model in NEST Simulator and run
The generated extension module can be loaded using ``nest.Install()``.

Using the PyNEST API, the model can be instantiated and simulated in NEST. The following code will create one instance of the neuron model (`nest.Create("izhikevich_tutorial")`), inject a constant current and run the simulation for 250 ms.

In [ ]:
nest.ResetKernel()
NESTTools.set_nest_verbosity("ALL")
nest.print_time = False    # print the time progress -- True might cause issues with Jupyter

# load dynamic library (the NESTML generated NEST extension module) into NEST kernel
nest.Install(module_name)

neuron = nest.Create(neuron_model_name)
voltmeter = nest.Create("voltmeter")

voltmeter.set({"record_from": ["v", "u"]})
nest.Connect(voltmeter, neuron)

cgs = nest.Create("dc_generator")
cgs.set({"amplitude": 25.})
nest.Connect(cgs, neuron)

sr = nest.Create("spike_recorder")
nest.Connect(neuron, sr)

nest.Simulate(250.)

spike_times = nest.GetStatus(sr, keys="events")[0]["times"]

fig, ax = plt.subplots(nrows=2)
ax[0].plot(voltmeter.get("events")["times"], voltmeter.get("events")["v"])
ax[1].plot(voltmeter.get("events")["times"], voltmeter.get("events")["u"])
ax[0].scatter(spike_times, 30 * np.ones_like(spike_times), marker="d", c="orange", alpha=.8, zorder=99)
for _ax in ax:
    _ax.grid(True)
ax[0].set_ylabel("v [mV]")
ax[1].set_ylabel("u")
ax[-1].set_xlabel("Time [ms]")
plt.show()
plt.close(fig)

## Adding a refractory mechanism

In order to model an absolute refractory state, in which the neuron cannot fire action potentials, different approaches can be used. In general, an extra parameter (say, ``refr_T``) is introduced, that defines the duration of the refractory period. A new state variable (say, ``refr_t``) can then act as a timer, counting the time of the refractory period that has already elapsed. The dynamics of ``refr_t`` could be specified in the ``update`` block, as follows:

```
   update:
       refr_t -= resolution()
```

The test for refractoriness can then be added in the ``onCondition`` block as follows:

```
   # if not refractory and threshold is crossed...
   onCondition(refr_t <= 0 ms and V_m > V_th):
       V_m = E_L    # Reset the membrane potential
       refr_t = refr_T    # Start the refractoriness timer
       emit_spike()
```

The disadvantage of this method is that it requires a call to the ``resolution()`` function, which is only supported by fixed-timestep simulators, and furthermore the timer is always counting, even when the neuron is not refractory anymore. To write the model in a more generic way, the refractoriness timer can alternatively be expressed as an ODE, which represents the timer in continuous-time, counting down to zero at a rate of one (milli)second per (milli)second:

```
   equations:
       refr_t' = -1
```

During the refractory period, the membrane potential should typically remain clamped to the reset or leak potential. It depends on the intended behavior of the model whether the synaptic currents and conductances also continue to be integrated or whether they are reset, and whether incoming spikes during the refractory period are taken into account or ignored.

In order to hold the membrane potential at the reset voltage during refractoriness, it can be simply excluded from the integration call:

```
   equations:
       I_syn' = ...
       V_m' = ...
       refr_t' = -1

   update:
       if refr_t > 0 ms:
           # neuron is absolute refractory, do not evolve V_m
           integrate_odes(I_syn, refr_t)
       else:
           # neuron not refractory
           integrate_odes(I_syn, V_m)
```

Note that in some cases, the finite resolution by which real numbers are expressed (as floating point numbers) in computers, can cause unexpected behaviors. If the simulation resolution is not exactly representable as a float (say, $\Delta t$ = 0.1 ms) then it could be the case that after 20 simulation steps, the timer has not reached zero, but a very small value very close to zero (say, 0.00000001 ms), causing the refractory period to end only in the next timestep. If this kind of behavior is undesired, the simulation resolution and refractory period can be chosen as powers of two (which can be represented exactly as floating points), or a small "epsilon" value can be included in the comparison in the model:

```
   parameters:
       float_epsilon ms = 1E-9 ms

   onCondition(refr_t <= float_epsilon ...):
       # ...
```

## Task

Modify the Izhikevich neuron model with your refractory mechanism of choice, and show that it has an upper firing rate limit by plotting the f-I curve.

## References

[1] Eugene M. Izhikevich, "Simple Model of Spiking Neurons", IEEE Transactions on Neural Networks, Vol. 14, No. 6, November 2003

## Copyright

This file is part of NEST.

Copyright (C) 2004 The NEST Initiative

NEST is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 2 of the License, or (at your option) any later version.

NEST is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with NEST.  If not, see <http://www.gnu.org/licenses/>.
